# Intelligent Agent

We created a `Agent` class in Week 2 notebook that builds an intelligent agent who can find a path from the upper-left corner of a grid to the lower-right corner. For this assignment, you are expected to modify this program so that it can find a path to an arbitrary position on the grid.

1. Modify the `__init__` method so that the `target`attribute records the correct target position, which is a list including the row index and column index.
2. Make sure that the `decide_action` method can still be used to find a path to the target position.
3. Update the `create_environment` method so that it can generate a grid with a given target position.
4. Run a simulation experiment with your agent. Display the path it takes to reach the target.

**Submission instructions:**
1. Write your programs in a new Colab notebook.
2. Run all programs to show execution results in the notebook.
3. Use the "Share" button to generate a sharable link of the notebook. Make sure to change the setting from "Restricted" to "Anyone with link".
4. Submit the link to Blackboard.

In [ ]:
import random

In [ ]:
# Try to create the function on your own.
def create_environment(num_rows, num_cols, num_obstacles, target=None):

    # 1. Create a 2D list represent an empty grid
    grid = []
    for row_index in range(num_rows):
        row = ["_"] * num_cols # The * operator creates multiple copies of an element
        grid.append(row)

    # 2. Indicate the starting position (upper-left) and the target position (lower-right)
    grid[0][0] = "O"
    # add this here for the target cell
    if target:
        target_row, target_col = target
    else:
        target_row, target_col = num_rows - 1, num_cols - 1

    grid[num_rows-1][num_cols-1] = "T"

    # 3. Randomly choose cells as obstacles
    count = 0
    while count < num_obstacles:
        row_index = random.randint(0, num_rows-1)
        col_index = random.randint(0, num_cols-1)
        # update this if statement to update how the obstacle works
        if grid[row_index][col_index] == "_" and [row_index, col_index] != [0,0] and [row_index, col_index] != [target_row, target_col]:
            grid[row_index][col_index] = "X"
            count += 1

    return grid

In [ ]:
class Agent:

    def __init__(self, grid, target=None):
        """
        Initialize important class variables
        """
        self.grid = grid
        self.position = [0, 0]
        self.num_rows = len(grid)
        self.num_cols = len(grid[0])
        self.target = target if target else [self.num_rows - 1, self.num_cols - 1]
        self.visited = [[0, 0]]
        self.path = [[0, 0]]

    def perceive_environment(self):
        """
        This method returns information of the up, down, left, and right cells adjacent to the agent.
        """
        # Look up
        row, col = self.position # The current row index and column index for the agent
        if row == 0:
          up = "Wall"
        elif self.grid[row-1][col] == "X":
          up = "Obstacle"
        else:
          up = "Empty"

        # Look down
        if row == self.num_rows - 1:
          down = "Wall"
        elif self.grid[row+1][col] == "X":
          down = "Obstacle"
        else:
          down = "Empty"

        # Look left
        if col == 0:
          left = "Wall"
        elif self.grid[row][col-1] == "X":
          left = "Obstacle"
        else:
          left = "Empty"

        # Look right
        if col == self.num_cols - 1:
          right = "Wall"
        elif self.grid[row][col+1] == "X":
          right = "Obstacle"
        else:
          right = "Empty"

        return up, down, left, right

    def decide_action(self):
        """
        Implement a decision-making process for the agent.
        Modifying this so that we choose the direction that
        moves closer to target
        """
        up, down, left, right = self.perceive_environment()

        row, col = self.position

        target_row, target_col = self.target

        directions = []

        # If the agent can still move to a new cell, do it
        # Here I will modify this code to check directions of
        # right down up left and calculate the distance to the target.
        # Will use abs so that we do not get a negative number when the agent is ahead of the target
        if right == "Empty" and [row, col+1] not in self.visited:
            distance = abs(row - target_row) + abs((col + 1) - target_col)
            directions.append(("Right", distance))
        if down == "Empty" and [row+1, col] not in self.visited:
            distance = abs((row + 1) - target_row) + abs(col - target_col)
            directions.append(("Down", distance))
        if up == "Empty" and [row-1, col] not in self.visited:
            distance = abs((row - 1) - target_row) + abs(col - target_col)
            directions.append(("Up", distance))
        if left == "Empty" and [row, col-1] not in self.visited:
            distance = abs(row - target_row) + abs((col - 1) - target_col)
            directions.append(("Left", distance))

        if directions:
            directions.sort(key=lambda x : x[1])
            return directions[0][0] # return smallest distance from this
        # If the agent has no new cell to move into: take a step back
        else:
            return "BackTrack"

    def take_action(self):
        """
        Update self.position, and check if the agent has reached the target.
        """
        action = self.decide_action()
        row, col = self.position # Current position

        if action == "Right":
            col += 1
        elif action == "Down":
            row += 1
        elif action == "Up":
            row -= 1
        elif action == "Left":
            col -= 1
        elif action == "BackTrack":
            if len(self.path) > 1:  # Check if path is not empty before popping
                self.path.pop()
                row, col = self.path[-1]
            else:
                # If the path is empty, the agent is stuck at the start or cannot move
                print("Agent is stuck.")
                return # Do not update position if stuck

        self.position = [row, col]

        # Before moving to a new position, add the current position to the visited list
        if self.position not in self.visited:
          self.visited.append(self.position)

        # Add the current position to the path if the agent is not back-tracking
        if action != "BackTrack":
            self.path.append(self.position)


In [ ]:
# Run a simulation experiment with your agent. Display the path it takes to reach the target.
rows, columns = 8, 8
target = [4, 6]

# set up a grid
grid = create_environment(rows, columns, num_obstacles=10, target=target)

# instantiate agent
agent = Agent(grid, target)

# set a limit of the amount of steps you'd like for better testing
steps = 0
max_steps = 500

# run it and display the path along with adding the limit of steps in while loop condition
while agent.position != agent.target and steps <= max_steps:
    agent.take_action()
    steps += 1

if steps >= max_steps:
    print("Went over amount of steps set!")

print("Path taken:")
print(agent.path)

Path taken:
[[0, 0], [0, 1], [0, 2], [0, 3], [0, 4], [0, 5], [0, 6], [1, 6], [2, 6], [3, 6], [4, 6]]
